# SSHafe Figures 1–6: Colab reproduction

This notebook rebuilds the six figures used in `sshafe.pdf` from the project notebooks and exports them as a six-page PDF. The figure mapping is:

1. Temporal patterns of SSH attempts
2. Distribution of nonzero per-second counts
3. 30-second sliding-window feature distributions
4. Top 20 LightGBM feature importances
5. ROC, precision–recall, and LightGBM confusion matrix
6. F1, precision, and recall by threshold, with the paper's 0.70 operating threshold

The source code is adapted from `assignment1_analysis.ipynb` (Figures 1–2) and `ssh_detector_more_accuracy.ipynb` (Figures 3–6). The existing project files are not needed in Colab; this notebook is self-contained.

**Run in Google Colab:** upload this notebook, choose **Runtime → Run all**, wait for the final download prompt, then save `sshafe_figures.pdf`. The public Kaggle dataset is downloaded by KaggleHub. Fixed seeds remove random-seed variation, but a changed upstream dataset or library/runtime version can still change the results.


## 1. Install the pinned libraries

The versions below are fixed to make Colab runs more consistent. The dataset itself is hosted externally, so keep the downloaded CSV files (or their checksums printed below) with your project if you need to audit a later rerun.


In [ ]:
%pip install -q kagglehub==0.3.12 lightgbm==4.6.0 scikit-learn==1.6.1 pandas==2.2.2 numpy==2.0.2 matplotlib==3.10.0 seaborn==0.13.2


## 2. Imports and deterministic settings


In [ ]:
import glob
import hashlib
import os
import random
from pathlib import Path

import kagglehub
import lightgbm as lgb
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_recall_curve, roc_auc_score, roc_curve
)
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print({
    'python': __import__('sys').version.split()[0],
    'pandas': pd.__version__, 'numpy': np.__version__,
    'matplotlib': mpl.__version__, 'lightgbm': lgb.__version__
})


## 3. Download and prepare the SSH anomaly dataset

The binary target follows the source notebooks: `normal` is class 0 and every other label is class 1. Rows are sorted by timestamp before the time-based holdout split.


In [ ]:
dataset_dir = kagglehub.dataset_download('mdwiraputradananjaya/ssh-anomaly-dataset')
csv_files = sorted(glob.glob(os.path.join(dataset_dir, '**', '*.csv'), recursive=True))
if not csv_files:
    raise FileNotFoundError(f'No CSV files found in KaggleHub dataset directory: {dataset_dir}')

print(f'Found {len(csv_files)} CSV file(s)')
for csv_path in csv_files:
    digest = hashlib.sha256(Path(csv_path).read_bytes()).hexdigest()
    print(f'{Path(csv_path).name}: sha256={digest}')

df = pd.concat([pd.read_csv(path) for path in csv_files], ignore_index=True)
required = {'timestamp', 'source_ip', 'username', 'event_type', 'status', 'label'}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f'Dataset is missing required columns: {sorted(missing)}')

df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)
df['label_bin'] = (df['label'].astype(str).str.strip().str.lower() != 'normal').astype(int)
print('Dataset shape:', df.shape)
print(df['label'].value_counts(dropna=False))
display(df.head())


## Figure 1 — Temporal patterns of SSH attempts

Counts are resampled to one-second bins across the full dataset time span.


In [ ]:
df_bf = df[df['label_bin'] == 1].set_index('timestamp')
df_normal = df[df['label_bin'] == 0].set_index('timestamp')
full_index = pd.date_range(df['timestamp'].min().floor('s'), df['timestamp'].max().ceil('s'), freq='1s')
ts_bf = df_bf.resample('1s').size().reindex(full_index, fill_value=0).rename('brute_force')
ts_normal = df_normal.resample('1s').size().reindex(full_index, fill_value=0).rename('normal')

fig1, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
axes[0].plot(ts_bf.index, ts_bf.values, color='tomato', alpha=0.7, linewidth=0.6, label='Brute Force')
axes[0].set_ylabel('Count/s'); axes[0].set_title('Raw Per-Second Counts'); axes[0].legend()
axes[1].plot(ts_normal.index, ts_normal.values, color='steelblue', alpha=0.7, linewidth=0.6, label='Normal')
axes[1].set_ylabel('Count/s'); axes[1].set_title('Normal Traffic'); axes[1].legend()
axes[2].plot(ts_bf.index, ts_bf.rolling(60).mean(), color='tomato', label='BF 60s MA')
axes[2].plot(ts_normal.index, ts_normal.rolling(60).mean(), color='steelblue', label='Normal 60s MA')
axes[2].set_ylabel('Count/s'); axes[2].set_title('60-Second Rolling Mean'); axes[2].legend()
fig1.suptitle('Temporal Patterns', fontsize=13)
fig1.tight_layout()
plt.show()


## Figure 2 — Distribution patterns of SSH attempts

This reproduces the two nonzero per-second count histograms shown in the PDF.


In [ ]:
fig2, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, series, name, color in [
    (axes[0], ts_bf, 'Brute Force', 'tomato'),
    (axes[1], ts_normal, 'Normal', 'steelblue'),
]:
    values = series.values
    ax.hist(values[values > 0], bins=50, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(f'{name} — Count Distribution (non-zero)')
    ax.set_xlabel('Events/sec'); ax.set_ylabel('Frequency')
fig2.suptitle('Distributional Analysis: Histograms', fontsize=13)
fig2.tight_layout()
plt.show()


## 4. Build sliding-window features

This uses the per-source-IP, inclusive trailing window feature logic from the model notebook. It can take several minutes in Colab because each event is evaluated against its IP's recent history.


In [ ]:
WINDOW_SIZES = [30, 60, 300]

def compute_ip_features(data, window_sec):
    data = data.copy()
    data['ts_epoch'] = data['timestamp'].astype('int64') / 1e9
    data['is_failed'] = (data['status'].astype(str).str.lower() != 'success').astype(int)
    data['is_success'] = (data['status'].astype(str).str.lower() == 'success').astype(int)
    records = []

    for _, group in data.groupby('source_ip', sort=False):
        group = group.sort_values('ts_epoch').reset_index()
        original_indices = group['index'].to_numpy()
        timestamps = group['ts_epoch'].to_numpy()
        failed = group['is_failed'].to_numpy()
        success = group['is_success'].to_numpy()
        usernames = group['username'].to_numpy()
        events = group['event_type'].astype(str).str.lower().to_numpy()

        for i, now in enumerate(timestamps):
            mask = (timestamps >= now - window_sec) & (timestamps <= now)
            n_attempts = int(mask.sum())
            n_failed = int(failed[mask].sum())
            n_success = int(success[mask].sum())
            fail_ratio = n_failed / n_attempts if n_attempts else 0
            unique_users = len(set(usernames[mask]))
            ts_window = timestamps[mask]
            if len(ts_window) > 1:
                iats = np.diff(np.sort(ts_window))
                iat_mean = iats.mean()
                iat_std = iats.std()
                iat_min = iats.min()
                iat_cv = iat_std / iat_mean if iat_mean > 0 else 0
            else:
                iat_mean = iat_std = iat_min = iat_cv = 0
            event_window = events[mask]
            records.append({
                'orig_idx': original_indices[i],
                f'w{window_sec}_n_attempts': n_attempts,
                f'w{window_sec}_attempt_rate': n_attempts / window_sec,
                f'w{window_sec}_fail_ratio': fail_ratio,
                f'w{window_sec}_unique_users': unique_users,
                f'w{window_sec}_iat_mean': iat_mean,
                f'w{window_sec}_iat_std': iat_std,
                f'w{window_sec}_iat_min': iat_min,
                f'w{window_sec}_iat_cv': iat_cv,
                f'w{window_sec}_n_accepted': sum('accept' in x for x in event_window),
                f'w{window_sec}_n_failed_pw': sum('fail' in x for x in event_window),
                f'w{window_sec}_n_invalid': sum('invalid' in x for x in event_window),
            })
    return pd.DataFrame(records).set_index('orig_idx').sort_index()

feature_frames = [compute_ip_features(df, window_sec=w) for w in WINDOW_SIZES]
features = pd.concat(feature_frames, axis=1)
features = features.loc[:, ~features.columns.duplicated()]
df_feat = df.join(features, how='left').dropna(subset=features.columns.tolist())
feat_cols = features.columns.tolist()
print('Feature matrix:', features.shape, '| aligned rows:', len(df_feat))


## Figure 3 — Features extracted from a 30-second sliding window


In [ ]:
key_features = [
    'w30_attempt_rate', 'w30_fail_ratio', 'w30_unique_users',
    'w30_iat_mean', 'w30_iat_cv', 'w30_n_attempts'
]
fig3, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, feature in enumerate(key_features):
    for label, color, name in [(0, 'steelblue', 'Normal'), (1, 'tomato', 'Brute Force')]:
        values = df_feat.loc[df_feat['label_bin'] == label, feature]
        values = values.clip(upper=df_feat[feature].quantile(0.99))
        axes[i].hist(values, bins=50, alpha=0.6, color=color, label=name, density=True)
    axes[i].set_title(feature); axes[i].legend()
    axes[i].set_xlabel('Value'); axes[i].set_ylabel('Density')
fig3.suptitle('Feature Distributions: Brute Force vs Normal (30s window)', fontsize=14, y=1.02)
fig3.tight_layout()
plt.show()


## 5. Time-ordered split and model training

The first 80% of timestamp-sorted feature rows are used for training; the final 20% are held out. The parameters follow the source model notebook, with a fixed seed.


In [ ]:
split_index = int(len(df_feat) * 0.8)
train_df = df_feat.iloc[:split_index]
test_df = df_feat.iloc[split_index:]
X_train = train_df[feat_cols].to_numpy()
y_train = train_df['label_bin'].to_numpy()
X_test = test_df[feat_cols].to_numpy()
y_test = test_df['label_bin'].to_numpy()

if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
    raise ValueError('Time-ordered split must contain both classes in train and test.')

scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
lgb_model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63, max_depth=-1,
    scale_pos_weight=scale_pos, subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgb_model.fit(X_train, y_train)
lgb_scores = lgb_model.predict_proba(X_test)[:, 1]
lgb_pred = lgb_model.predict(X_test)

# Isolation Forest baseline for the ROC and precision-recall panels in Figure 5.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
iso_model = IsolationForest(n_estimators=200, contamination=0.05, random_state=SEED, n_jobs=-1)
iso_model.fit(X_train_scaled[y_train == 0])
iso_scores = -iso_model.score_samples(X_test_scaled)

print('Train rows:', len(train_df), '| holdout rows:', len(test_df))
print('LightGBM ROC-AUC:', round(roc_auc_score(y_test, lgb_scores), 6))
print('LightGBM F1:', round(f1_score(y_test, lgb_pred), 6))


## Figure 4 — Top 20 LightGBM feature importances


In [ ]:
importance = pd.Series(lgb_model.feature_importances_, index=feat_cols).sort_values(ascending=True).tail(20)
fig4, ax = plt.subplots(figsize=(10, 7))
importance.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('LightGBM Feature Importance (Top 20)')
ax.set_xlabel('Importance')
fig4.tight_layout()
plt.show()


## Figure 5 — Model performance and confusion matrix


In [ ]:
fig5, axes = plt.subplots(1, 3, figsize=(18, 5))
for scores, name, color in [(iso_scores, 'Isolation Forest', 'orange'), (lgb_scores, 'LightGBM', 'steelblue')]:
    fpr, tpr, _ = roc_curve(y_test, scores)
    axes[0].plot(fpr, tpr, color=color, label=f'{name} (AUC={roc_auc_score(y_test, scores):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title('ROC Curve'); axes[0].legend()

for scores, name, color in [(iso_scores, 'Isolation Forest', 'orange'), (lgb_scores, 'LightGBM', 'steelblue')]:
    precision, recall, _ = precision_recall_curve(y_test, scores)
    ap = average_precision_score(y_test, scores)
    axes[1].plot(recall, precision, color=color, label=f'{name} (AP={ap:.3f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].set_title('Precision Recall Curve'); axes[1].legend()

cm = confusion_matrix(y_test, lgb_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Normal', 'Brute Force'], yticklabels=['Normal', 'Brute Force'])
axes[2].set_title('Confusion Matrix — LightGBM')
axes[2].set_ylabel('True'); axes[2].set_xlabel('Predicted')
fig5.suptitle('Model Comparison', fontsize=14)
fig5.tight_layout()
plt.show()


## Figure 6 — Fine-tuning the decision threshold

The PDF reports a 0.70 threshold. This figure shows the threshold curves on the reproduced holdout scores and marks 0.70 so the operating point matches the paper.


In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, lgb_scores)
f1_values = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
REPORTED_THRESHOLD = 0.70

fig6, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, f1_values, color='steelblue', label='F1')
ax.plot(thresholds, precision[:-1], color='green', alpha=0.6, label='Precision')
ax.plot(thresholds, recall[:-1], color='tomato', alpha=0.6, label='Recall')
ax.axvline(REPORTED_THRESHOLD, color='black', linestyle='--', label=f'Best threshold={REPORTED_THRESHOLD:.3f}')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('LightGBM: Threshold vs F1 / Precision / Recall')
ax.legend(); fig6.tight_layout()
plt.show()

threshold_idx = int(np.argmin(np.abs(thresholds - REPORTED_THRESHOLD)))
print(f'F1 at the nearest evaluated threshold to 0.70: {f1_values[threshold_idx]:.6f}')


## 6. Export Figures 1–6 as a PDF

The output is a six-page PDF with one numbered figure per page. In Colab, the final line downloads the PDF to your computer.


In [ ]:
from google.colab import files

figures = [fig1, fig2, fig3, fig4, fig5, fig6]

for number, fig in enumerate(figures, start=1):
    output_pdf = Path(f"/content/sshafe_figure_{number}.pdf")
    fig.savefig(output_pdf, format="pdf", bbox_inches="tight")
    print(f"Created: {output_pdf} ({output_pdf.stat().st_size:,} bytes)")
    files.download(str(output_pdf))